In [1]:
import re

def extract_additive_data(text):
    """
    Extract manual and AI identified additives for each entry
    """
    results = []

    # Split by entry separators
    entries = text.split('=' * 60 + '\n#')

    for entry in entries:
        if not entry.strip():
            continue

        entry_data = {}

        # Extract Entry ID
        entry_id_match = re.search(r'Entry ID:\s*(\d+)', entry)
        if entry_id_match:
            entry_data['entry_id'] = int(entry_id_match.group(1))
        else:
            continue  # Skip if no entry ID found

        # Extract manual additives (from ORIGINAL INGREDIENT LIST)
        manual_match = re.search(r'🧾 ORIGINAL INGREDIENT LIST:(.*?)(?=🎯|$)', entry, re.DOTALL)
        if manual_match:
            manual_text = manual_match.group(1).strip()
            # Handle both comma and semicolon delimiters
            manual_text = manual_text.replace(';', ',')  # Convert semicolons to commas first
            # Split by commas and clean up
            manual_additives = [item.strip() for item in manual_text.split(',') if item.strip()]
            # Convert to set with lowercase for comparison
            entry_data['manual_additives'] = set(additive.lower().strip() for additive in manual_additives)
            entry_data['manual_additives_original'] = manual_additives  # Keep original case for reference
        else:
            entry_data['manual_additives'] = set()
            entry_data['manual_additives_original'] = []

        # Extract AI identified additives
        ai_match = re.search(r'🔍 IDENTIFIED ADDITIVES \((\d+)\):(.*?)(?=🤖|$)', entry, re.DOTALL)
        if ai_match:
            ai_count = int(ai_match.group(1))
            ai_text = ai_match.group(2).strip()
            # Extract numbered list items
            ai_items = re.findall(r'\d+\.\s*([^\n]+)', ai_text)
            # Convert to set with lowercase for comparison
            entry_data['ai_additives'] = set(additive.lower().strip() for additive in ai_items)
            entry_data['ai_additives_original'] = ai_items  # Keep original case for reference
            entry_data['ai_count_stated'] = ai_count
        else:
            entry_data['ai_additives'] = set()
            entry_data['ai_additives_original'] = []
            entry_data['ai_count_stated'] = 0

        results.append(entry_data)

    return results


with open('/content/drive/MyDrive/Food additives RAG+LLM/2.10.2 and 2.10.3/Manual check of identified additives_full.txt', 'r') as file:
    experiment_text = file.read()
# Extract the data
extracted_data = extract_additive_data(experiment_text)

# Display results
for i, entry in enumerate(extracted_data):
    print(f"\n--- Entry {i+1} ---")
    print(f"Entry ID: {entry['entry_id']}")
    print(f"Manual additives count: {len(entry['manual_additives'])}")
    print(f"AI additives count: {len(entry['ai_additives'])} (stated: {entry['ai_count_stated']})")
    assert len(entry['ai_additives'])==entry['ai_count_stated']
    print(f"Manual additives: {sorted(list(entry['manual_additives']))}")
    print(f"AI additives: {sorted(list(entry['ai_additives']))}")


--- Entry 1 ---
Entry ID: 4792
Manual additives count: 19
AI additives count: 16 (stated: 16)
Manual additives: ['artificial flavor', 'calcium propionate', 'carbohydrate gum', 'cinnamon', 'datem', 'dextrose', 'folic acid', 'high fructose corn syrup', 'iron', 'monoglycerides', 'natural flavor', 'niacin', 'riboflavin', 'silicon dioxide', 'sodium aluminum phosphate', 'sodium bicarbonate', 'soy lecithin', 'thiamine mononitrate', 'yeast']
AI additives: ['artificial flavor', 'artificial flavors', 'calcium propionate', 'carbohydrate gum', 'cinnamon', 'datem', 'dextrose', 'high fructose corn syrup', 'mono- and diglycerides', 'natural flavor', 'niacin', 'riboflavin', 'sodium benzoate', 'soy lecithin', 'thiamine mononitrate', 'yeast']

--- Entry 2 ---
Entry ID: 41998
Manual additives count: 5
AI additives count: 5 (stated: 5)
Manual additives: ['citric acid', 'lactose', 'maltodextrin', 'sodium citrate', 'sodium diacetate']
AI additives: ['citric acid', 'lactose', 'maltodextrin', 'sodium citrate

In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from sklearn.metrics import precision_score, recall_score, f1_score

# Set style for better plots
plt.style.use('default')
sns.set_palette("husl")

def parse_validation_string(validation_string):
    """
    Parse the validation results string and extract data for each entry
    """
    entries = []

    # Split by entry markers
    entry_blocks = re.split(r'--- Entry \d+ ---', validation_string)[1:]  # Skip first empty split

    for i, block in enumerate(entry_blocks, 1):
        try:
            # Extract Entry ID
            entry_id_match = re.search(r'Entry ID: (\d+)', block)
            entry_id = int(entry_id_match.group(1)) if entry_id_match else i

            # Extract counts
            manual_count_match = re.search(r'Manual additives count: (\d+)', block)
            ai_count_match = re.search(r'AI additives count: (\d+)', block)

            manual_count = int(manual_count_match.group(1)) if manual_count_match else 0
            ai_count = int(ai_count_match.group(1)) if ai_count_match else 0

            # Extract additives lists
            manual_match = re.search(r"Manual additives: (\[.*?\])", block)
            ai_match = re.search(r"AI additives: (\[.*?\])", block)

            if manual_match and ai_match:
                # Clean and parse the lists
                manual_str = manual_match.group(1).replace("'", '"')
                ai_str = ai_match.group(1).replace("'", '"')

                manual_additives = eval(manual_str)
                ai_additives = eval(ai_str)

                entries.append({
                    'entry_id': entry_id,
                    'manual_count': manual_count,
                    'ai_count': ai_count,
                    'manual_additives': manual_additives,
                    'ai_additives': ai_additives
                })

        except Exception as e:
            print(f"Error parsing entry {i}: {e}")
            continue

    return entries

def normalize_additive_name(name):
    """
    Normalize additive names for better matching
    """
    name = name.lower().strip()

    # Handle common variations
    normalizations = {
        'mono- and diglycerides': 'monoglycerides',
        'mono and diglycerides': 'monoglycerides',
        # === FLAVOR VARIATIONS ===
        'artificial flavors': 'artificial flavor',
        'artificial flavoring': 'artificial flavor',
        'artificial flavouring': 'artificial flavor',
        'natural flavors': 'natural flavor',
        'natural flavoring': 'natural flavor',
        'natural flavouring': 'natural flavor',
        'natural and artificial flavors': 'natural and artificial flavor',
        'natural and artificial flavoring': 'natural and artificial flavor',
        'natural and artificial flavouring': 'natural and artificial flavor',
        'natural & artificial flavors': 'natural and artificial flavor',
        'natural & artificial flavor': 'natural and artificial flavor',

        # === COLOR VARIATIONS ===
        'artificial colors': 'artificial color',
        'artificial coloring': 'artificial color',
        'artificial colouring': 'artificial color',
        'artificial colour': 'artificial color',
        'artificial colours': 'artificial color',
        'natural colors': 'natural color',
        'natural coloring': 'natural color',
        'natural colouring': 'natural color',
        'natural colour': 'natural color',
        'natural colours': 'natural color',
        'natural and artificial colors': 'natural and artificial color',
        'natural and artificial coloring': 'natural and artificial color',
        'natural & artificial colors': 'natural and artificial color'
        }

    return normalizations.get(name, name)

def calculate_metrics_per_entry(manual_additives, ai_additives):
    """
    Calculate precision, recall, F1 for a single entry
    """
    # Normalize names for comparison
    manual_normalized = set(normalize_additive_name(a) for a in manual_additives)
    ai_normalized = set(normalize_additive_name(a) for a in ai_additives)

    # Calculate metrics
    true_positives = len(manual_normalized & ai_normalized)
    false_positives = len(ai_normalized - manual_normalized)
    false_negatives = len(manual_normalized - ai_normalized)

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'manual_normalized': manual_normalized,
        'ai_normalized': ai_normalized
    }

def analyze_validation_results(validation_string):
    """
    Complete analysis of validation results
    """
    # Parse the data
    entries = parse_validation_string(validation_string)
    print(f"Successfully parsed {len(entries)} entries")

    # Calculate metrics for each entry
    results = []
    all_false_positives = []
    all_false_negatives = []

    for entry in entries:
        metrics = calculate_metrics_per_entry(entry['manual_additives'], entry['ai_additives'])

        result = {
            'entry_id': entry['entry_id'],
            'manual_count': entry['manual_count'],
            'ai_count': entry['ai_count'],
            **metrics
        }

        results.append(result)

        # Collect error types
        if metrics['false_positives'] > 0:
            fps = metrics['ai_normalized'] - metrics['manual_normalized']
            all_false_positives.extend(fps)

        if metrics['false_negatives'] > 0:
            fns = metrics['manual_normalized'] - metrics['ai_normalized']
            all_false_negatives.extend(fns)

    # Convert to DataFrame
    df = pd.DataFrame(results)

    # Calculate overall metrics
    total_tp = df['true_positives'].sum()
    total_fp = df['false_positives'].sum()
    total_fn = df['false_negatives'].sum()

    overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    overall_f1 = 2 * (overall_precision * overall_recall) / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0

    # Summary statistics
    summary = {
        'total_entries': len(entries),
        'overall_precision': overall_precision,
        'overall_recall': overall_recall,
        'overall_f1': overall_f1,
        'mean_precision': df['precision'].mean(),
        'mean_recall': df['recall'].mean(),
        'mean_f1': df['f1'].mean(),
        'std_precision': df['precision'].std(),
        'std_recall': df['recall'].std(),
        'std_f1': df['f1'].std(),
        'total_manual_additives': df['manual_count'].sum(),
        'total_ai_additives': df['ai_count'].sum(),
        'total_true_positives': total_tp,
        'total_false_positives': total_fp,
        'total_false_negatives': total_fn
    }

    return df, summary, all_false_positives, all_false_negatives

def create_visualizations(df, summary, false_positives, false_negatives):
    """
    Create and save comprehensive visualizations as separate files
    """
    # 1. Metrics distribution
    plt.figure(figsize=(10, 6))
    plt.hist([df['precision'], df['recall'], df['f1']],
                   bins=20, alpha=0.7, label=['Precision', 'Recall', 'F1-Score'])
    plt.title('Distribution of Metrics Across Entries')
    plt.xlabel('Score')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('metrics_distribution.png', bbox_inches='tight')
    plt.close()

    # 2. Count comparison
    plt.figure(figsize=(8, 8))
    plt.scatter(df['manual_count'], df['ai_count'], alpha=0.6)
    plt.plot([0, df['manual_count'].max()], [0, df['manual_count'].max()], 'r--', alpha=0.8)
    plt.title('Manual vs AI Additive Counts')
    plt.xlabel('Manual Count')
    plt.ylabel('AI Count')
    plt.grid(True, alpha=0.3)
    plt.savefig('manual_vs_ai_counts.png', bbox_inches='tight')
    plt.close()


    # 3. Error types
    plt.figure(figsize=(8, 8))
    error_data = [summary['total_true_positives'], summary['total_false_positives'], summary['total_false_negatives']]
    error_labels = ['True Positives', 'False Positives', 'False Negatives']
    plt.pie(error_data, labels=error_labels, autopct='%1.1f%%', startangle=90)
    plt.title('Distribution of Classification Results')
    plt.savefig('classification_distribution.png', bbox_inches='tight')
    plt.close()


    # 4. Metrics by entry count
    plt.figure(figsize=(10, 6))
    plt.scatter(df['manual_count'], df['f1'], alpha=0.6, label='F1-Score')
    plt.scatter(df['manual_count'], df['precision'], alpha=0.6, label='Precision')
    plt.scatter(df['manual_count'], df['recall'], alpha=0.6, label='Recall')
    plt.title('Metrics vs Manual Additive Count')
    plt.xlabel('Manual Additive Count')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('metrics_vs_manual_count.png', bbox_inches='tight')
    plt.close()


    # 5. Most common false positives
    plt.figure(figsize=(10, 6))
    fp_counter = Counter(false_positives)
    if fp_counter:
        fp_top = fp_counter.most_common(10)
        fp_names, fp_counts = zip(*fp_top)
        plt.barh(range(len(fp_names)), fp_counts)
        plt.yticks(range(len(fp_names)), fp_names)
        plt.title('Most Common False Positives')
        plt.xlabel('Frequency')
        plt.savefig('most_common_false_positives.png', bbox_inches='tight')
        plt.close()


    # 6. Most common false negatives
    plt.figure(figsize=(10, 6))
    fn_counter = Counter(false_negatives)
    if fn_counter:
        fn_top = fn_counter.most_common(10)
        fn_names, fn_counts = zip(*fn_top)
        plt.barh(range(len(fn_names)), fn_counts)
        plt.yticks(range(len(fn_names)), fn_names)
        plt.title('Most Common False Negatives')
        plt.xlabel('Frequency')
        plt.savefig('most_common_false_negatives.png', bbox_inches='tight')
        plt.close()


def print_summary_report(summary, false_positives, false_negatives):
    """
    Print formatted summary report
    """
    print("="*80)
    print("FOOD ADDITIVE IDENTIFICATION VALIDATION RESULTS")
    print("="*80)

    print(f"\n📊 OVERALL PERFORMANCE METRICS")
    print(f"Total Entries Analyzed: {summary['total_entries']}")
    print(f"Overall Precision: {summary['overall_precision']:.3f}")
    print(f"Overall Recall: {summary['overall_recall']:.3f}")
    print(f"Overall F1-Score: {summary['overall_f1']:.3f}")

    print(f"\n📈 AVERAGE PERFORMANCE (Per Entry)")
    print(f"Mean Precision: {summary['mean_precision']:.3f} ± {summary['std_precision']:.3f}")
    print(f"Mean Recall: {summary['mean_recall']:.3f} ± {summary['std_recall']:.3f}")
    print(f"Mean F1-Score: {summary['mean_f1']:.3f} ± {summary['std_f1']:.3f}")

    print(f"\n🔢 CLASSIFICATION BREAKDOWN")
    print(f"Total Manual Additives: {summary['total_manual_additives']}")
    print(f"Total AI Additives: {summary['total_ai_additives']}")
    print(f"True Positives: {summary['total_true_positives']}")
    print(f"False Positives: {summary['total_false_positives']}")
    print(f"False Negatives: {summary['total_false_negatives']}")

    print(f"\n❌ ERROR ANALYSIS")
    fp_counter = Counter(false_positives)
    fn_counter = Counter(false_negatives)

    print(f"Most Common False Positives (AI identified, but not manual):")
    for additive, count in fp_counter.most_common(5):
        print(f"  • {additive}: {count} times")

    print(f"\nMost Common False Negatives (Manual identified, but AI missed):")
    for additive, count in fn_counter.most_common(5):
        print(f"  • {additive}: {count} times")

# Example usage function
def run_analysis(validation_string):
    """
    Run complete analysis pipeline
    """
    df, summary, false_positives, false_negatives = analyze_validation_results(validation_string)

    # Print summary report
    print_summary_report(summary, false_positives, false_negatives)

    # Create visualizations
    create_visualizations(df, summary, false_positives, false_negatives)

    # Return detailed results for further analysis
    return df, summary, false_positives, false_negatives

# Instructions for use:
print("="*80)
print("FOOD ADDITIVE VALIDATION ANALYSIS TOOL")
print("="*80)
print("\nTo use this tool:")
print("1. Copy your validation results string into a variable called 'validation_data'")
print("2. Run: df, summary, fps, fns = run_analysis(validation_data)")
print("3. The tool will generate comprehensive analysis and visualizations")
print("\nExample:")
print("validation_data = '''--- Entry 1 --- Entry ID: 4792 ... [your full string] ...'''")
print("df, summary, fps, fns = run_analysis(validation_data)")

FOOD ADDITIVE VALIDATION ANALYSIS TOOL

To use this tool:
1. Copy your validation results string into a variable called 'validation_data'
2. Run: df, summary, fps, fns = run_analysis(validation_data)
3. The tool will generate comprehensive analysis and visualizations

Example:
validation_data = '''--- Entry 1 --- Entry ID: 4792 ... [your full string] ...'''
df, summary, fps, fns = run_analysis(validation_data)


In [2]:
with open('/content/drive/MyDrive/Food additives RAG+LLM/2.10.2 and 2.10.3/2.10.2 results to analyze.txt', 'r') as file:
    validation_data =  file.read()
df, summary, fps, fns = run_analysis(validation_data)

Successfully parsed 108 entries
FOOD ADDITIVE IDENTIFICATION VALIDATION RESULTS

📊 OVERALL PERFORMANCE METRICS
Total Entries Analyzed: 108
Overall Precision: 0.728
Overall Recall: 0.788
Overall F1-Score: 0.757

📈 AVERAGE PERFORMANCE (Per Entry)
Mean Precision: 0.744 ± 0.250
Mean Recall: 0.808 ± 0.209
Mean F1-Score: 0.749 ± 0.201

🔢 CLASSIFICATION BREAKDOWN
Total Manual Additives: 833
Total AI Additives: 900
True Positives: 652
False Positives: 244
False Negatives: 175

❌ ERROR ANALYSIS
Most Common False Positives (AI identified, but not manual):
  • sodium benzoate: 33 times
  • citric acid: 15 times
  • xanthan gum: 13 times
  • red 40: 11 times
  • corn starch: 11 times

Most Common False Negatives (Manual identified, but AI missed):
  • iron: 15 times
  • folic acid: 9 times
  • baking soda: 8 times
  • natural flavor: 8 times
  • soy lecithin: 7 times
